In [ ]:
%cd /content
!rm -rf coding-exercises
!git clone https://github.com/eth-bmai-fs26/coding-exercises.git
%cd coding-exercises
!git checkout cvae_sanzi
%cd "CVAE CX"
!pip install -q datasets

In [ ]:
from utils import *
from models import *
from data import *
import torch.nn.functional as F
import os

setup()

X_train, attrs_train, X_test, attrs_test = load_celeba(max_samples=30000)

In [ ]:
class CelebAConvCVAE(nn.Module):
    """
    Faithful PyTorch translation of EleMisi/ConditionalVAE.
    Encoder: (43, 64, 64) -> mu, log_var  (latent_dim each)
    Decoder: (latent_dim + 40) -> (3, 64, 64)
    """
    def __init__(self, latent_dim=128, label_dim=40):
        super().__init__()
        self.latent_dim = latent_dim
        self.label_dim  = label_dim

        self.encoder_conv = nn.Sequential(
            nn.Conv2d(3 + label_dim, 32,  3, stride=2, padding=1),
            nn.BatchNorm2d(32),  nn.LeakyReLU(0.2),
            nn.Conv2d(32,  64,  3, stride=2, padding=1),
            nn.BatchNorm2d(64),  nn.LeakyReLU(0.2),
            nn.Conv2d(64,  128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
        )
        self.fc_latent = nn.Linear(256 * 4 * 4, latent_dim * 2)

        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim + label_dim, 256 * 4 * 4),
            nn.LeakyReLU(0.2),
        )
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(256, 256, 3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(256, 128, 3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(128, 64,  3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(64),  nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(64,  32,  3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(32),  nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32,   3,  3, stride=1, padding=1),
            nn.Sigmoid(),
        )

    def conditional_input(self, x, label):
        label_map = label.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, x.size(2), x.size(3))
        return torch.cat([x, label_map], dim=1)

    def reparametrize(self, mu, log_var):
        return mu + torch.exp(0.5 * log_var) * torch.randn_like(log_var)

    def encode(self, x, label):
        h   = self.encoder_conv(self.conditional_input(x, label)).flatten(1)
        out = self.fc_latent(h)
        return out[:, :self.latent_dim], out[:, self.latent_dim:]

    def decode(self, z, label):
        h = self.decoder_fc(torch.cat([z, label], dim=1))
        return self.decoder_conv(h.view(-1, 256, 4, 4))

    def forward(self, x, label):
        mu, log_var = self.encode(x, label)
        return self.decode(self.reparametrize(mu, log_var), label), mu, log_var


def cvae_loss(recon_x, x, mu, log_var, beta=1.0):
    batch_size = x.size(0)
    recon_loss = F.binary_cross_entropy(recon_x, x, reduction='sum') / batch_size
    kl_loss    = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / batch_size
    return recon_loss + beta * kl_loss, recon_loss, kl_loss

In [ ]:
LATENT_DIM = 128
EPOCHS     = 100

beta_configs = [
    (0.1, 'celeba_cvae_beta0.10_recon_dominant'),
    (0.6, 'celeba_cvae_beta0.60_recon_favoured'),
    (0.8, 'celeba_cvae_beta0.80_slight_recon_bias'),
    (1.0, 'celeba_cvae_beta1.00_balanced'),
    (1.2, 'celeba_cvae_beta1.20_slight_kl_bias'),
    (1.5, 'celeba_cvae_beta1.50_kl_favoured'),
    (5.0, 'celeba_cvae_beta5.00_kl_dominant'),
]

os.makedirs('pretrained_models', exist_ok=True)

for beta, name in beta_configs:
    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print(f"{'='*60}")

    model = CelebAConvCVAE(latent_dim=LATENT_DIM).to(DEVICE)
    model, history = train_cvae_celeba(
        model, cvae_loss, X_train, attrs_train,
        epochs=EPOCHS, lr=1e-3, beta=beta
    )

    save_path = f'pretrained_models/{name}.pt'
    torch.save({
        'model_state_dict': model.state_dict(),
        'latent_dim':       LATENT_DIM,
        'beta':             beta,
        'epochs':           EPOCHS,
        'loss_history':     history,
    }, save_path)
    print(f'Saved -> {save_path}')

In [ ]:
# Optional: persist models to Google Drive so they survive session restarts
#
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# dest = '/content/drive/MyDrive/cvae_pretrained_models'
# shutil.copytree('pretrained_models', dest, dirs_exist_ok=True)
# print(f'Models copied to {dest}')